# YOLOv8 Nano Model Training for License Plate Detection
Run the cells below sequentially to install requirements, download your dataset, validate it, and train your model.

In [1]:
!pip install ultralytics pyyaml kaggle torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.5 MB/s eta 0:00:00


In [7]:
import os
import sys
import yaml
import zipfile
import shutil
from pathlib import Path
from ultralytics import YOLO

# ============================================================================
# CONFIGURATION SECTION
# ============================================================================

# Absolute path to the dataset directory
DATASET_DIR = os.path.abspath("C:\\Users\\bss10\\Desktop\\drishti\\drishti_kavach\\archive\\IDDDetectionsYOLODataset")

# Kaggle dataset identifier
KAGGLE_DATASET = "redzapdos123/indian-driving-dataset-detections-yolov11"

# Training hyperparameters
EPOCHS = 10           # Number of training epochs
BATCH_SIZE = 8        # Batch size (CPU-friendly; use 16 if you have sufficient RAM)
IMG_SIZE = 640        # Image size for training
DEVICE = "cpu"        # Training device (CPU-only for Raspberry Pi compatibility)
MODEL = "yolov8n.pt"  # YOLOv8 nano model

In [12]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d redzapdos123/indian-driving-dataset-detections-yolov11 --unzip -p /content/dataset

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/redzapdos123/indian-driving-dataset-detections-yolov11
License(s): CC-BY-NC-SA-4.0
100% 20.5G/20.5G [03:49<00:00, 95.9MB/s]



In [14]:
import os
import yaml

dataset_root = "/content/dataset"
yaml_path = None

# Search for data.yaml in all subdirectories
for root, dirs, files in os.walk(dataset_root):
    if "data.yaml" in files:
        yaml_path = os.path.join(root, "data.yaml")
        break

if yaml_path is None:
    print("Error: data.yaml not found. Check if the download succeeded.")
else:
    print(f"Found data.yaml at: {yaml_path}")

    # Determine the actual dataset folder
    yaml_dir = os.path.dirname(yaml_path)

    # Read the file
    with open(yaml_path, "r") as file:
        data = yaml.safe_load(file)

    # Update the paths using absolute locations
    data["train"] = os.path.join(yaml_dir, "train/images")
    data["val"] = os.path.join(yaml_dir, "val/images")
    data["test"] = os.path.join(yaml_dir, "test/images")

    # Save the file
    with open(yaml_path, "w") as file:
        yaml.dump(data, file, default_flow_style=False)

    print("Updated paths successfully.")

Found data.yaml at: /content/dataset/IDDDetectionsYOLODataset/data.yaml
Updated paths successfully.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=yaml_path,
    epochs=10,
    batch=8,
    imgsz=640,
    device="cpu"
)

Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/IDDDetectionsYOLODataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mas

In [ ]:
from pathlib import Path

# Define output directories
run_dir = Path("runs/detect/license_plate_detection")
best_weights = run_dir / "weights/best.pt"
last_weights = run_dir / "weights/last.pt"

print("="*70)
print("Training Complete!")
print("="*70)

# Verify and print the best weights location
if best_weights.exists():
    print(f"Best weights saved at:\n  {best_weights.absolute()}")
else:
    print("Best weights file not found. Check the training logs above.")

# Verify and print the last checkpoint location
if last_weights.exists():
    print(f"\nLast checkpoint saved at:\n  {last_weights.absolute()}")

# Print the master directory
print(f"\nAll training artifacts saved at:\n  {run_dir.absolute()}")
print("="*70)